# Negative Binomial GLM — Arrival and Within Trip Counts

Extends the departure NB model to `arrival_count` and `within_count`.
Uses the same time-based train/test split as `Predictive_model.ipynb`
(train: years < 2025, test: 2025), with `YEAR` kept numeric (not categorical)
so the model can predict on a held-out year not seen during fitting.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

df = pd.read_csv('./Data/model_df_with_event_flag.csv')

# Time-based split: train on everything before 2025, test on 2025
model_df_clean = df[df['YEAR'] != 2020].copy()

train = model_df_clean[model_df_clean['YEAR'] < 2025].copy()
test = model_df_clean[model_df_clean['YEAR'] == 2025].copy()

train['YEAR_c'] = train['YEAR'] - 2019
test['YEAR_c'] = test['YEAR'] - 2019

print(f"Train: {len(train)} rows, Test: {len(test)} rows")

Train: 8059 rows, Test: 2207 rows


In [2]:
def fit_and_evaluate_nb(target, train, test):
    formula = (
        f"{target} ~ event_flag + avg_temp + avg_precip "
        "+ C(MONTH, Treatment(reference='Nov')) + YEAR_c + C(HOUR) + C(day_of_week)"
    )

    model = smf.negativebinomial(formula, data=train).fit(disp=0, maxiter=200, method='bfgs')
    print(f"Converged: {model.mle_retvals['converged']}")

    preds = model.predict(test)
    mae = mean_absolute_error(test[target], preds)
    rmse = root_mean_squared_error(test[target], preds)
    r2 = r2_score(test[target], preds)

    print(f"--- NB GLM: {target} ---")
    print(f"  Estimated alpha: {model.params['alpha']:.4f}")
    print(f"  MAE:  {mae:.1f}")
    print(f"  RMSE: {rmse:.1f}")
    print(f"  R2:   {r2:.3f}\n")

    return model, {'target': target, 'mae': mae, 'rmse': rmse, 'r2': r2}

nb_model_dep, dep_results = fit_and_evaluate_nb('departure_count', train, test)
print(nb_model_dep.summary())

Converged: True
--- NB GLM: departure_count ---
  Estimated alpha: 0.1107
  MAE:  243.4
  RMSE: 330.4
  R2:   0.590

                     NegativeBinomial Regression Results                      
Dep. Variable:        departure_count   No. Observations:                 8059
Model:               NegativeBinomial   Df Residuals:                     8023
Method:                           MLE   Df Model:                           35
Date:                Sun, 30 Aug 2026   Pseudo R-squ.:                 0.06517
Time:                        17:50:49   Log-Likelihood:                -58757.
converged:                       True   LL-Null:                       -62853.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                  

## Arrival count

In [3]:
nb_model_arr, arr_results = fit_and_evaluate_nb('arrival_count', train, test)
print(nb_model_arr.summary())

Converged: True
--- NB GLM: arrival_count ---
  Estimated alpha: 0.0937
  MAE:  213.2
  RMSE: 290.0
  R2:   0.587

                     NegativeBinomial Regression Results                      
Dep. Variable:          arrival_count   No. Observations:                 8059
Model:               NegativeBinomial   Df Residuals:                     8023
Method:                           MLE   Df Model:                           35
Date:                Sun, 30 Aug 2026   Pseudo R-squ.:                 0.08297
Time:                        17:50:49   Log-Likelihood:                -57433.
converged:                       True   LL-Null:                       -62629.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                    

## Within count

In [4]:
nb_model_within, within_results = fit_and_evaluate_nb('within_count', train, test)
print(nb_model_within.summary())

Converged: True
--- NB GLM: within_count ---
  Estimated alpha: 0.1316
  MAE:  47.5
  RMSE: 67.3
  R2:   0.567

                     NegativeBinomial Regression Results                      
Dep. Variable:           within_count   No. Observations:                 8059
Model:               NegativeBinomial   Df Residuals:                     8023
Method:                           MLE   Df Model:                           35
Date:                Sun, 30 Aug 2026   Pseudo R-squ.:                 0.09839
Time:                        17:50:49   Log-Likelihood:                -44849.
converged:                       True   LL-Null:                       -49744.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                       

## Departure count

In [5]:
nb_model_dep, dep_results = fit_and_evaluate_nb('departure_count', train, test)
print(nb_model_dep.summary())

Converged: True
--- NB GLM: departure_count ---
  Estimated alpha: 0.1107
  MAE:  243.4
  RMSE: 330.4
  R2:   0.590

                     NegativeBinomial Regression Results                      
Dep. Variable:        departure_count   No. Observations:                 8059
Model:               NegativeBinomial   Df Residuals:                     8023
Method:                           MLE   Df Model:                           35
Date:                Sun, 30 Aug 2026   Pseudo R-squ.:                 0.06517
Time:                        17:50:49   Log-Likelihood:                -58757.
converged:                       True   LL-Null:                       -62853.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                  

## Combined results table

In [6]:
results_df = pd.DataFrame([arr_results, within_results, dep_results])
results_df

,target,mae,rmse,r2
0,arrival_count,213.168637,289.953256,0.586750
1,within_count,47.517753,67.312804,0.566931
2,departure_count,243.443367,330.431610,0.590102
